# Q5 Correlation Analysis (Battery Cycle Life)

## Analysis Goal
Use **first 100 cycles** to identify features related to battery lifetime.

Steps:
1. Load data
2. Convert MATLAB structure
3. Feature engineering (first 100 cycles)
4. Correlation analysis
5. Multicollinearity check
6. Model strategy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import mat73
import scipy.io as sio

from statsmodels.stats.outliers_influence import variance_inflation_factor

## Data Load Function

In [ ]:
def load_mat(path):

    try:
        data = mat73.loadmat(path)
        print("Loaded:",path.name)

    except:

        data = sio.loadmat(
            path,
            simplify_cells=True
        )

        print("Loaded:",path.name)

    return data

## Convert MATLAB batch to cell rows

In [ ]:
def extract_cells(batch):

    cells=[]

    n=len(batch['cycle_life'])

    for i in range(n):

        cell={}

        for key in batch.keys():

            value=batch[key]

            if isinstance(value,list):

                cell[key]=value[i]

            else:

                cell[key]=value

        cells.append(cell)

    return cells

## Feature Engineering (First 100 cycles)

In [ ]:
EARLY_CYCLE = 100


def early_mean(arr):

    arr=np.array(arr).reshape(-1)

    if len(arr)==0:
        return np.nan

    return np.mean(arr[:EARLY_CYCLE])


def early_delta(arr):

    arr=np.array(arr).reshape(-1)

    if len(arr)<2:
        return np.nan

    arr=arr[:EARLY_CYCLE]

    return arr[-1]-arr[0]


def build_feature_df(cells):

    rows=[]

    for cell in cells:

        summary=cell['summary']

        row={

        'cycle_life':cell['cycle_life'],

        'IR_mean':early_mean(summary['IR']),

        'QDischarge_mean':early_mean(summary['QDischarge']),

        'Tmax_mean':early_mean(summary['Tmax']),

        'Tavg_mean':early_mean(summary['Tavg']),

        'charge_time_mean':early_mean(summary['chargetime']),

        'IR_delta':early_delta(summary['IR']),

        'QDischarge_delta':early_delta(summary['QDischarge'])

        }

        rows.append(row)

    return pd.DataFrame(rows)

## Load Dataset

In [ ]:
DATA_DIR=Path("../data/data-30")

b1=load_mat(DATA_DIR/"2017-05-12_batchdata_updated_struct_errorcorrect.mat")
b2=load_mat(DATA_DIR/"2018-02-20_batchdata_updated_struct_errorcorrect.mat")
b3=load_mat(DATA_DIR/"2018-04-12_batchdata_updated_struct_errorcorrect.mat")

## Create Cells

In [ ]:
cells1=extract_cells(b1['batch'])
cells2=extract_cells(b2['batch'])
cells3=extract_cells(b3['batch'])

print(len(cells1))
print(len(cells2))
print(len(cells3))

## Feature Tables

In [ ]:
df1=build_feature_df(cells1)
df2=build_feature_df(cells2)
df3=build_feature_df(cells3)

df1['batch']='B1'
df2['batch']='B2'
df3['batch']='B3'

df=pd.concat([df1,df2,df3])

df.head()

## Correlation Analysis

In [ ]:
features=[

'cycle_life',
'IR_mean',
'QDischarge_mean',
'Tmax_mean',
'Tavg_mean',
'charge_time_mean',
'IR_delta',
'QDischarge_delta'

]

corr=df[features].corr()

corr

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
corr,
annot=True,
cmap='coolwarm'
)

plt.title("Correlation Matrix (First 100 cycles)")

plt.show()

## Strongest Feature

In [ ]:
target_corr=corr['cycle_life']

target_corr=target_corr.drop('cycle_life')

target_corr.sort_values()

In [ ]:
best_feature=target_corr.abs().idxmax()

best_value=target_corr[best_feature]

print(best_feature)
print(best_value)

## Multicollinearity

In [ ]:
X=df[[
'IR_mean',
'QDischarge_mean',
'Tmax_mean',
'Tavg_mean',
'charge_time_mean',
'IR_delta',
'QDischarge_delta'
]].dropna()

plt.figure(figsize=(8,6))

sns.heatmap(
X.corr(),
annot=True,
cmap='coolwarm'
)

plt.show()

In [ ]:
vif=pd.DataFrame()

vif['feature']=X.columns

vif['VIF']=[

variance_inflation_factor(X.values,i)

for i in range(X.shape[1])

]

vif.sort_values('VIF',ascending=False)